# Step 1 baseline assessment maps

Creates the same two-map baseline/protected-area figure as `p3_fig_1_baseline_map.ipynb`, but uses the updated area statistics and replaces the 2013 mangrove class with the Forces of Nature mangrove layer.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def find_dphil_root(start_path=None):
    search_start = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate_path in [search_start, *search_start.parents]:
        if (candidate_path / 'dphil_paper_3').exists() and (candidate_path / 'dphil_common_cross_cutting').exists():
            return candidate_path
        nested_dphil_root = candidate_path / 'dphil_papers'
        if (nested_dphil_root / 'dphil_paper_3').exists() and (nested_dphil_root / 'dphil_common_cross_cutting').exists():
            return nested_dphil_root
    raise FileNotFoundError('Could not locate dphil_papers from the current working directory.')


dphil_root = find_dphil_root()
robyn_libraries_path = dphil_root / 'robyns_libraries'
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))

import Robyn_paper_2_defs

pd.options.display.max_rows = 100
pd.options.display.max_columns = 50
plt.rcParams['font.family'] = 'Times New Roman'

JAMAICA_METRIC_CRS = 'EPSG:3448'


In [ ]:
def next_available_path(output_path):
    if not output_path.exists():
        return output_path
    for suffix_number in range(2, 1000):
        candidate_path = output_path.with_name(f'{output_path.stem}_v{suffix_number}{output_path.suffix}')
        if not candidate_path.exists():
            return candidate_path
    raise FileExistsError(f'Could not find an unused filename for {output_path}')


paper_root = dphil_root / 'dphil_paper_3'
common_root = dphil_root / 'dphil_common_cross_cutting'
area_extents_dir = paper_root / 'processed_data' / 'baseline_assessment' / 'area_extents'
figure_output_dir = paper_root / 'results' / 'figures'
figure_output_dir.mkdir(parents=True, exist_ok=True)

input_paths = {
    'jamaica_boundary': common_root / 'common_incoming_data' / 'boundaries' / 'jamaica.gpkg',
    'landcover_2013': common_root / 'common_incoming_data' / 'landcover' / '2013_landcover' / '2013_landuse_LandCover.shp',
    'forces_of_nature_mangroves': paper_root / 'inputs' / 'forces_of_nature_mangroves' / 'mangroves.shp',
    'forest_reserves': common_root / 'common_incoming_data' / 'protected_landcover' / 'Forest_reserves.shp',
    'protected_areas': common_root / 'common_incoming_data' / 'protected_landcover' / 'Protected_areas.shp',
    'coral_reefs': paper_root / 'processed_data' / 'corals' / 'corals_clipped_1000m.shp',
    'seagrass': paper_root / 'processed_data' / 'seagrass' / 'seagrass_clipped_10000m.shp',
    'ecosystem_area_summary': area_extents_dir / 'ecosystem_area_extents_with_protected_status.csv',
}

missing_paths = {name: path for name, path in input_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(missing_paths)


In [ ]:
def load_valid_layer(path):
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f'Layer has no CRS: {path}')
    layer = layer.to_crs(JAMAICA_METRIC_CRS)
    layer = layer[~layer.geometry.is_empty].copy()
    layer = layer[layer.geometry.notna()].copy()
    layer['geometry'] = layer.geometry.make_valid()
    return layer[~layer.geometry.is_empty].copy()


def format_map_axis(axis, map_bounds, padding_fraction=0.025):
    min_x, min_y, max_x, max_y = map_bounds
    padding = max(max_x - min_x, max_y - min_y) * padding_fraction
    axis.set_xlim(min_x - padding, max_x + padding)
    axis.set_ylim(min_y - padding, max_y + padding)
    axis.set_aspect('equal')
    axis.set_axis_off()
    for spine in axis.spines.values():
        spine.set_visible(False)


def format_area_ha(area_ha):
    if 0 < area_ha < 10:
        return f'{area_ha:.1f} ha'
    return f'{area_ha:,.0f} ha'


def format_percentage(percentage):
    if 0 < percentage < 0.005:
        return '<0.01%'
    return f'{percentage:.1f}%'


In [ ]:
landcover_category_mapping = {
    'Bare Rock': 'Bare Rock',
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
    'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
    'Herbaceous Wetland': 'Freshwater wetland',
    'Mangrove Forest': 'Mangrove',
    'Fields: Bare Land': 'Agriculture',
    'Open dry forest - Short': 'Open dry forest',
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest',
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
    'Quarry': 'Bauxite extraction / quarry',
    'Water Body': 'Water body',
    'Buildings and other infrastructures': 'Buildings and other infrastructure',
    'Fields and Secondary Forest': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
    'Bauxite Extraction': 'Bauxite extraction / quarry',
    'Disturbed broadleaved forest (Secondary Forest)': 'Secondary forest',
    'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
    'Bamboo and Secondary Forest': 'Mixed land use: agriculture and bamboo',
    'Hardwood Plantation: Euculytus': 'Plantation',
    'Hardwood Plantation: Mixed': 'Plantation',
    'Swamp Forest': 'Swamp forest',
    'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Hardwood Plantation: Mahoe': 'Plantation',
    'Hardwood Plantation: Mahogany': 'Plantation',
    'Bamboo': 'Bamboo',
    'Closed broadleaved forest (Primary Forest)': 'Primary forest',
    'Secondary Forest': 'Secondary forest',
}

category_colors = {
    'Primary forest': '#00441B',
    'Secondary forest': '#238B45',
    'Open dry forest': '#A4C639',
    'Mangrove': '#008080',
    'Swamp forest': '#6B8E23',
    'Freshwater wetland': '#4169E1',
    'Water body': '#4682B4',
    'Agriculture': '#8B4513',
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',
    'Mixed land use: agriculture and bamboo': '#D2B48C',
    'Plantation': '#F5DEB3',
    'Bamboo': '#F4A460',
    'Bauxite extraction / quarry': '#B22222',
    'Bare Rock': '#A9A9A9',
    'Buildings and other infrastructure': '#000000',
}

map_category_order = [
    'Primary forest',
    'Secondary forest',
    'Open dry forest',
    'Mangrove',
    'Swamp forest',
    'Freshwater wetland',
    'Water body',
    'Agriculture',
    'Mixed land use: forests with bamboo or agriculture/plantation',
    'Mixed land use: agriculture and bamboo',
    'Plantation',
    'Bamboo',
    'Bauxite extraction / quarry',
    'Bare Rock',
    'Buildings and other infrastructure',
]

marine_colors = {
    'Coral reefs': '#FFC0CB',
    'Seagrass': '#40E0D0',
}


In [ ]:
ecosystem_area_summary = pd.read_csv(input_paths['ecosystem_area_summary'])
marine_area_summary = ecosystem_area_summary[ecosystem_area_summary['domain'].eq('marine')].copy()


In [ ]:
jamaica_boundary = load_valid_layer(input_paths['jamaica_boundary'])
landcover_2013 = load_valid_layer(input_paths['landcover_2013']).dropna(subset=['Classify']).copy()
unknown_landcover_classes = sorted(set(landcover_2013['Classify']) - set(landcover_category_mapping))
if unknown_landcover_classes:
    raise ValueError(f'Unmapped land-cover classes: {unknown_landcover_classes}')

forces_of_nature_mangroves = load_valid_layer(input_paths['forces_of_nature_mangroves'])
forces_of_nature_mangrove_geometry = forces_of_nature_mangroves.geometry.union_all()

landcover_without_2013_mangrove = landcover_2013[~landcover_2013['Classify'].eq('Mangrove Forest')].copy()
landcover_overlapping_fon_mangroves = landcover_without_2013_mangrove.geometry.intersects(forces_of_nature_mangrove_geometry)
landcover_without_2013_mangrove.loc[landcover_overlapping_fon_mangroves, 'geometry'] = (
    landcover_without_2013_mangrove.loc[landcover_overlapping_fon_mangroves].geometry.difference(forces_of_nature_mangrove_geometry)
)
landcover_without_2013_mangrove['geometry'] = landcover_without_2013_mangrove.geometry.make_valid()
landcover_without_2013_mangrove = landcover_without_2013_mangrove[
    ~landcover_without_2013_mangrove.geometry.is_empty
].copy()
landcover_without_2013_mangrove = landcover_without_2013_mangrove[
    landcover_without_2013_mangrove.geometry.notna()
].copy()
landcover_without_2013_mangrove['category'] = landcover_without_2013_mangrove['Classify'].replace(landcover_category_mapping)

forces_of_nature_mangrove_landcover = gpd.GeoDataFrame(
    {'category': ['Mangrove']},
    geometry=[forces_of_nature_mangrove_geometry],
    crs=JAMAICA_METRIC_CRS,
)

terrestrial_landcover = gpd.GeoDataFrame(
    pd.concat(
        [
            landcover_without_2013_mangrove[['category', 'geometry']],
            forces_of_nature_mangrove_landcover[['category', 'geometry']],
        ],
        ignore_index=True,
    ),
    geometry='geometry',
    crs=JAMAICA_METRIC_CRS,
)
terrestrial_landcover['color'] = terrestrial_landcover['category'].map(category_colors)

missing_colors = sorted(set(terrestrial_landcover['category']) - set(category_colors))
if missing_colors:
    raise ValueError(f'Missing colours for mapped categories: {missing_colors}')


In [ ]:
forest_reserves = load_valid_layer(input_paths['forest_reserves'])
protected_areas = load_valid_layer(input_paths['protected_areas'])
protected_network_parts = gpd.GeoDataFrame(
    pd.concat([forest_reserves[['geometry']], protected_areas[['geometry']]], ignore_index=True),
    geometry='geometry',
    crs=JAMAICA_METRIC_CRS,
)
protected_network = gpd.GeoDataFrame(geometry=[protected_network_parts.geometry.union_all()], crs=JAMAICA_METRIC_CRS)

protected_landcover = gpd.clip(terrestrial_landcover[['category', 'geometry']].copy(), protected_network)
protected_landcover['color'] = protected_landcover['category'].map(category_colors)

coral_reefs = load_valid_layer(input_paths['coral_reefs'])
seagrass = load_valid_layer(input_paths['seagrass'])
protected_coral_reefs = gpd.clip(coral_reefs, protected_network)
protected_seagrass = gpd.clip(seagrass, protected_network)

terrestrial_area_summary = (
    terrestrial_landcover.assign(area_m2=terrestrial_landcover.geometry.area)
    .groupby('category', as_index=False)['area_m2']
    .sum()
)
protected_area_summary = (
    protected_landcover.assign(protected_area_m2=protected_landcover.geometry.area)
    .groupby('category', as_index=False)['protected_area_m2']
    .sum()
)
terrestrial_area_summary = terrestrial_area_summary.merge(protected_area_summary, on='category', how='left')
terrestrial_area_summary['protected_area_m2'] = terrestrial_area_summary['protected_area_m2'].fillna(0)
terrestrial_area_summary['area_ha'] = terrestrial_area_summary['area_m2'] / 1e4
terrestrial_area_summary['percentage_protected'] = (
    terrestrial_area_summary['protected_area_m2'] / terrestrial_area_summary['area_m2'] * 100
)
category_order_lookup = {category: category_order for category_order, category in enumerate(map_category_order)}
terrestrial_area_summary['category_order'] = terrestrial_area_summary['category'].map(category_order_lookup)
terrestrial_area_summary = terrestrial_area_summary.sort_values('category_order').reset_index(drop=True)

marine_area_summary = marine_area_summary[['category', 'area_ha', 'percentage_protected']].copy()

mapped_terrestrial_area_ha = terrestrial_area_summary['area_ha'].sum()
terrestrial_area_summary['percentage_of_mapped_terrestrial_area'] = (
    terrestrial_area_summary['area_ha'] / mapped_terrestrial_area_ha * 100
)

baseline_legend_handles = [
    Patch(
        facecolor=category_colors[summary_row['category']],
        edgecolor='none',
        label=(
            f"{summary_row['category']} "
            f"({format_area_ha(summary_row['area_ha'])}; "
            f"{format_percentage(summary_row['percentage_of_mapped_terrestrial_area'])})"
        ),
    )
    for summary_row in terrestrial_area_summary.to_dict('records')
]
baseline_legend_handles.extend([
    Patch(
        facecolor=marine_colors[summary_row['category']],
        edgecolor='none',
        label=f"{summary_row['category']} ({format_area_ha(summary_row['area_ha'])})",
    )
    for summary_row in marine_area_summary.to_dict('records')
])
baseline_legend_handles.append(
    Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica Boundary')
)

protected_legend_handles = [
    Patch(
        facecolor=category_colors[summary_row['category']],
        edgecolor='none',
        label=f"{summary_row['category']} ({format_percentage(summary_row['percentage_protected'])} protected)",
    )
    for summary_row in terrestrial_area_summary.to_dict('records')
]
protected_legend_handles.extend([
    Patch(
        facecolor=marine_colors[summary_row['category']],
        edgecolor='none',
        label=f"{summary_row['category']} ({format_percentage(summary_row['percentage_protected'])} protected)",
    )
    for summary_row in marine_area_summary.to_dict('records')
])
protected_legend_handles.append(
    Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica Boundary')
)

map_bounds = pd.concat(
    [
        terrestrial_landcover.geometry.bounds,
        coral_reefs.geometry.bounds,
        seagrass.geometry.bounds,
        jamaica_boundary.geometry.bounds,
    ],
    ignore_index=True,
).agg({'minx': 'min', 'miny': 'min', 'maxx': 'max', 'maxy': 'max'}).to_numpy()


In [ ]:
def add_legend_axis(legend_axis, legend_handles, legend_title, column_count=4):
    legend_axis.set_axis_off()
    legend_axis.legend(
        handles=legend_handles,
        title=legend_title,
        loc='center',
        ncol=column_count,
        frameon=False,
        fontsize=9,
        title_fontsize=10,
        columnspacing=1.4,
        handlelength=1.8,
        handletextpad=0.5,
    )


def add_orientation_markers(axis):
    Robyn_paper_2_defs.draw_north_arrow(
        axis,
        location=(0.86, 0.84),
        size=0.04,
        fontsize=8,
        label_offset=0.02,
    )
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        length_km=20,
        location=(0.86, 0.76),
        linewidth=1.0,
        tick_height=0.008,
        label_offset=0.015,
        km_offset=0.008,
    )


def plot_baseline_map(axis):
    terrestrial_landcover.plot(ax=axis, color=terrestrial_landcover['color'], linewidth=0.1, edgecolor='black')
    coral_reefs.plot(ax=axis, color=marine_colors['Coral reefs'], edgecolor=marine_colors['Coral reefs'], alpha=0.8, linewidth=1)
    seagrass.plot(ax=axis, color=marine_colors['Seagrass'], edgecolor=marine_colors['Seagrass'], alpha=0.6, linewidth=1)
    jamaica_boundary.plot(ax=axis, facecolor='none', edgecolor='black', linewidth=1, linestyle='--', alpha=0.8)
    format_map_axis(axis, map_bounds)


def plot_protected_map(axis):
    protected_landcover.plot(ax=axis, color=protected_landcover['color'], linewidth=0.1, edgecolor='black')
    protected_coral_reefs.plot(ax=axis, color=marine_colors['Coral reefs'], edgecolor=marine_colors['Coral reefs'], alpha=0.8, linewidth=1)
    protected_seagrass.plot(ax=axis, color=marine_colors['Seagrass'], edgecolor=marine_colors['Seagrass'], alpha=0.6, linewidth=1)
    jamaica_boundary.plot(ax=axis, facecolor='none', edgecolor='black', linewidth=1, linestyle='--', alpha=0.8)
    format_map_axis(axis, map_bounds)


In [ ]:
baseline_map_path = next_available_path(figure_output_dir / 'fig_2a_baseline_landcover_map.png')
protected_map_path = next_available_path(figure_output_dir / 'fig_2b_protected_landcover_map.png')

baseline_figure, baseline_axes = plt.subplots(
    2,
    1,
    figsize=(20, 8.5),
    dpi=300,
    gridspec_kw={'height_ratios': [1, 0.24]},
)
baseline_map_axis = baseline_axes[0]
baseline_legend_axis = baseline_axes[1]
plot_baseline_map(baseline_map_axis)
add_orientation_markers(baseline_map_axis)
add_legend_axis(baseline_legend_axis, baseline_legend_handles, 'Land cover extent (ha; % terrestrial for land classes)', column_count=4)
baseline_figure.tight_layout(h_pad=0.1)
baseline_figure.savefig(baseline_map_path, dpi=300, bbox_inches='tight')
plt.close(baseline_figure)

protected_figure, protected_axes = plt.subplots(
    2,
    1,
    figsize=(20, 8.5),
    dpi=300,
    gridspec_kw={'height_ratios': [1, 0.24]},
)
protected_map_axis = protected_axes[0]
protected_legend_axis = protected_axes[1]
plot_protected_map(protected_map_axis)
add_orientation_markers(protected_map_axis)
add_legend_axis(protected_legend_axis, protected_legend_handles, 'Protected coverage', column_count=4)
protected_figure.tight_layout(h_pad=0.1)
protected_figure.savefig(protected_map_path, dpi=300, bbox_inches='tight')
plt.close(protected_figure)

print(f'Saved baseline map: {baseline_map_path}')
print(f'Saved protected map: {protected_map_path}')


In [ ]:
panel_map_path = next_available_path(figure_output_dir / 'fig_2_baseline_protected_panel.png')

panel_figure, panel_axes = plt.subplots(
    4,
    1,
    figsize=(20, 15),
    dpi=300,
    gridspec_kw={'height_ratios': [1, 0.22, 1, 0.24]},
)
baseline_panel_axis = panel_axes[0]
baseline_panel_legend_axis = panel_axes[1]
protected_panel_axis = panel_axes[2]
protected_panel_legend_axis = panel_axes[3]

plot_baseline_map(baseline_panel_axis)
add_orientation_markers(baseline_panel_axis)
add_legend_axis(baseline_panel_legend_axis, baseline_legend_handles, 'Land cover extent (ha; % terrestrial for land classes)', column_count=4)
plot_protected_map(protected_panel_axis)
add_legend_axis(protected_panel_legend_axis, protected_legend_handles, 'Protected coverage', column_count=4)

panel_figure.tight_layout(h_pad=0.1)
panel_figure.savefig(panel_map_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved panel map: {panel_map_path}')
